# Sanity Check - Step 03: Bad Channels Detect

Überprüft:
- Bad channels identifiziert
- QC-Reports erstellt
- Markierung in Raw-Objekten
- Statistiken plausibel

In [ ]:
import sys
from pathlib import Path
import mne

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

## 1. Bad Channels Detected Files laden

In [ ]:
subject_id = config.SUBJECTS[0]

files = {}
for person in ["P1", "P2"]:
    path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_badchannels_detected.fif"
    if path.exists():
        files[person] = mne.io.read_raw_fif(str(path), preload=False)
        print(f"✓ {person}: {path.name}")
    else:
        print(f"✗ {person}: File not found")

## 2. Markierte Bad Channels

In [ ]:
for person, raw in files.items():
    bads = raw.info.get('bads', [])
    
    print(f"\n=== {person} ===")
    print(f"Bad channels: {len(bads)}")
    if bads:
        print(f"  Kanäle: {', '.join(bads)}")
    else:
        print(f"  Keine Bad Channels markiert")

## 3. QC-Reports

In [ ]:
for person in ["P1", "P2"]:
    report_path = config.QC_DIR / f"sub-{subject_id}_{person}_bad_channels_detect.tsv"
    
    if report_path.exists():
        print(f"\n✓ {person}: {report_path.name}")
        with open(report_path, 'r') as f:
            lines = f.readlines()
            print(f"  Analyzeiert: {len(lines)-1} Kanäle")
            # Show first few lines
            print(f"  Header: {lines[0].strip()}")
            if len(lines) > 1:
                print(f"  Erste Zeile: {lines[1].strip()[:60]}...")
    else:
        print(f"\n✗ {person}: QC-Report nicht gefunden")

## 4. Sanity Checks

In [ ]:
for person, raw in files.items():
    print(f"\n{person}:")
    
    bads = raw.info.get('bads', [])
    total_eeg = len(mne.pick_types(raw.info, eeg=True))
    
    print(f"  Total EEG channels: {total_eeg}")
    print(f"  Bad channels: {len(bads)}")
    
    if len(bads) > total_eeg:
        print(f"  ✗ Fehler: Mehr Bad-Channels als EEG-Kanäle!")
    elif len(bads) == 0:
        print(f"  ✓ Keine Bad-Channels (oder alle passieren den Test)")
    else:
        print(f"  ✓ {len(bads)} Bad-Channels erkannt (plausibel)")